# Temporal Spectral Embedding — sklearn-backed

Spectral embedding (graph-Laplacian eigenmaps) of 20-day windows of HAR-OLS
residuals on the real 30-min RV series. The heavy lifting — k-NN affinity
graph, normalized Laplacian, bottom-d eigenvectors — is delegated to
`sklearn.manifold.SpectralEmbedding`. The only custom code is a thin
`SpectralBasis` wrapper that gives `MultiStageBacktest` the
`.phi_train + .embed(v_test)` interface it expects.

The notebook is the source of truth: the `# export` cells below are
auto-exported to `src/features/extractors/spectral_embedding.py`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO = Path.cwd()
while REPO.parent != REPO and not (REPO / "src").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.linear_model import Ridge

from src.backtest.executor import _build_har_and_calendar, load_and_transform
from src.features.transforms.residualizer import Residualizer
from src.features.transforms.scaling import rolling_robust_scale
from src.features.transforms.target import PERIODS_PER_DAY

## 1. Load + HAR-prep the real data

Same data prep as the spectral_knn pipeline: `load_and_transform` with
diurnal-adjusted RV target, 240-period winsorization, NaN-drop; then
`_build_har_and_calendar` for the 13 features. Target is `adj_RV` shifted
by `horizon` rows.

In [ ]:
HORIZON = 1

df, _ = load_and_transform(
    "data", exog_cols=[],
    target_use_diurnal=True, target_winsor_window=240, dropna_with_exog=True,
)
df, feature_names = _build_har_and_calendar(df, exog_cols=[], add_calendar=True)
df["target"] = df["adj_RV"].shift(-HORIZON)
df = df.dropna(subset=["target"] + feature_names).reset_index(drop=True)
df["t"] = pd.to_datetime(df["t"])

print(f"rows:             {len(df):,}")
print(f"date range:       {df['t'].min()}  ..  {df['t'].max()}")
print(f"feature columns:  {len(feature_names)}  -> {feature_names}")

## 2. Whole-series rolling-robust-scale + Residualizer

A single `Residualizer(Ridge)` fit on the first `WARMUP_DAYS` of pre-scaled
data, then OOS residuals on the rest. `Residualizer` is the same class the
`spectral_knn` pipeline uses inside `MultiStageBacktest` — the notebook
shares the residual computation with the production backtest.

A real walk-forward refits the residualizer every step; here one fit is
enough for the embedding visualization.

In [ ]:
WARMUP_DAYS = 500
train_win = WARMUP_DAYS * PERIODS_PER_DAY
RIDGE_ALPHA = 1.0

X = df[feature_names].to_numpy(dtype=np.float64)
y = df["target"].to_numpy(dtype=np.float64)

X_scaled = rolling_robust_scale(X, train_win)

# Use the same Residualizer the spectral_knn pipeline uses inside MultiStageBacktest.
res = Residualizer(lambda: Ridge(alpha=RIDGE_ALPHA))
res.fit(X_scaled[:train_win], y[:train_win])
residuals = res.residuals(X_scaled[train_win:], y[train_win:])
dates_resid = df["t"].iloc[train_win:].reset_index(drop=True)

print(f"residuals: {residuals.shape}    OOS region: {dates_resid.iloc[0]}  ..  {dates_resid.iloc[-1]}")
print(f"R^2 OOS (Ridge baseline on HAR features): {1 - residuals.var() / y[train_win:].var():.4f}")

In [ ]:
# Residual series with named crisis windows.
fig, ax = plt.subplots(figsize=(13, 2.6))
ax.plot(dates_resid, residuals, lw=0.3, c="k", alpha=0.6)
ax.axhline(0, c="gray", lw=0.5)
for label, lo, hi, color in [
    ("GFC", "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt", "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue", "2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon", "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID", "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23", "2023-03-08", "2023-03-22", "tab:green"),
]:
    ax.axvspan(pd.Timestamp(lo), pd.Timestamp(hi), alpha=0.18, color=color, label=label)
ax.legend(fontsize=7, ncol=6, loc="upper right")
ax.set_title("Ridge-OOS residuals over time, with marked crisis windows")
ax.set_ylim(np.quantile(residuals, [0.001, 0.999]))
plt.tight_layout(); plt.show()

## 3. The spectral embedding machinery (sklearn-backed)

Cells below are marked `# export` and define what `MultiStageBacktest`
consumes. `build_embedding` delegates the graph + Laplacian + eigendecomp
to `sklearn.manifold.SpectralEmbedding`; the `SpectralBasis` wrapper adds
out-of-sample extension via `sklearn.neighbors.NearestNeighbors` (sklearn's
spectral embedding doesn't ship `.transform()`).

Trade-off vs. a hand-rolled implementation:
* `affinity='nearest_neighbors'` uses **binary** k-NN edges (no Gaussian
  weighting). Acceptable for an exploration baseline; if you need Gaussian
  weights, pass a precomputed affinity matrix instead.
* sklearn's `SpectralEmbedding` doesn't expose the eigenvalues. Trade
  introspection for one-line setup.

In [ ]:
# export
"""Temporal spectral embedding (sklearn-backed).

A thin wrapper over :class:`sklearn.manifold.SpectralEmbedding` that gives
:class:`~src.backtest.multi_stage.MultiStageBacktest` the
``.phi_train + .embed(v_test)`` interface it expects. Out-of-sample
extension is a weighted-kNN Nyström over the training views.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from sklearn.manifold import SpectralEmbedding
from sklearn.neighbors import NearestNeighbors

In [ ]:
# export
@dataclass
class SpectralBasis:
    """Frozen training-side spectral embedding state.

    ``sklearn.manifold.SpectralEmbedding`` doesn't implement ``transform()``,
    so we cache a ``NearestNeighbors`` index over the training views to do
    weighted-kNN Nyström extension for new test points.
    """

    views_train: np.ndarray
    phi_train: np.ndarray
    k_graph: int
    nn_index: NearestNeighbors

    def embed(self, v_test: np.ndarray) -> np.ndarray:
        """Embed a single test view via weighted-kNN Nyström. Returns shape (d,)."""
        dists, idx = self.nn_index.kneighbors(v_test[None, :], n_neighbors=self.k_graph)
        dists = dists.ravel()
        idx = idx.ravel()
        sigma = float(np.median(dists)) + 1e-12
        w = np.exp(-(dists**2) / (2 * sigma**2))
        s = w.sum()
        if s <= 0:
            return self.phi_train[idx].mean(axis=0)
        w = w / s
        return (w[:, None] * self.phi_train[idx]).sum(axis=0)

    def embed_batch(self, V_test: np.ndarray) -> np.ndarray:
        """Embed a batch of test views. Returns shape (M, d). Vectorized."""
        dists, idx = self.nn_index.kneighbors(V_test, n_neighbors=self.k_graph)
        sigma = np.median(dists, axis=1, keepdims=True) + 1e-12
        w = np.exp(-(dists**2) / (2 * sigma**2))
        w = w / np.clip(w.sum(axis=1, keepdims=True), 1e-12, None)
        return np.einsum("mk,mkd->md", w, self.phi_train[idx])

In [ ]:
# export
def build_embedding(views: np.ndarray, d: int, k_graph: int, seed: int = 42) -> SpectralBasis:
    """Spectral embedding of temporal views.

    Delegates to :class:`sklearn.manifold.SpectralEmbedding` with binary
    ``affinity='nearest_neighbors'`` (k = ``k_graph``) and ARPACK
    eigensolver. The fitted ``NearestNeighbors`` index over ``views`` is
    cached on the returned :class:`SpectralBasis` so out-of-sample test
    points can be embedded via Nyström without re-fitting anything.
    """
    spectral = SpectralEmbedding(
        n_components=d,
        affinity="nearest_neighbors",
        n_neighbors=k_graph,
        random_state=seed,
    )
    phi_train = spectral.fit_transform(views)
    nn_index = NearestNeighbors(n_neighbors=k_graph).fit(views)
    return SpectralBasis(
        views_train=views,
        phi_train=phi_train,
        k_graph=k_graph,
        nn_index=nn_index,
    )

## 4. Form views + build the embedding

Each view is `W=960` consecutive residuals — 20 trading days. Subsample to
one view per day (every 48 bars) so the eigendecomp on ~4-5k points stays
fast (~few seconds). `build_embedding` is the function defined in the
machinery section above.

In [ ]:
VIEW_WINDOW = 960               # 20 days * 48 bars
SUBSAMPLE_STEP = PERIODS_PER_DAY  # one view per day
EMBEDDING_DIM = 8
GRAPH_K = 10

view_idx = np.arange(VIEW_WINDOW, len(residuals), SUBSAMPLE_STEP)
views = np.stack([residuals[i - VIEW_WINDOW : i] for i in view_idx])
view_dates = dates_resid.iloc[view_idx].reset_index(drop=True)

print(f"views shape: {views.shape}   (N views, W bars each)")
print(f"date span:   {view_dates.iloc[0]}  ..  {view_dates.iloc[-1]}")

In [ ]:
%time basis = build_embedding(views, d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)

phi = basis.phi_train
print(f"phi: {phi.shape}")

## 5. What does the embedding actually see?

Three colorings of the same 2D scatter `phi_1` vs `phi_2`:

* **By year** — does the embedding cluster epochs? GFC vs the 2010s low-vol era?
* **By view RMS** — vol level proxy. Gradient = embedding sorts by amplitude;
  non-gradient = it sorts by *shape* of the residual trajectory.
* **By crisis-window membership** — overlay the same 6 named windows from
  the residual plot. Crisis-window dots clustering is the useful signal.

In [ ]:
view_years = view_dates.dt.year.to_numpy()
view_rms = np.sqrt((views**2).mean(axis=1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc0 = axes[0].scatter(phi[:, 0], phi[:, 1], c=view_years, s=8, alpha=0.7, cmap="viridis")
axes[0].set_xlabel("phi_1"); axes[0].set_ylabel("phi_2")
axes[0].set_title("colored by year")
plt.colorbar(sc0, ax=axes[0], label="year")

sc1 = axes[1].scatter(phi[:, 0], phi[:, 1], c=view_rms, s=8, alpha=0.7,
                       cmap="plasma", norm=LogNorm())
axes[1].set_xlabel("phi_1"); axes[1].set_ylabel("phi_2")
axes[1].set_title("colored by view RMS (vol amplitude proxy, log)")
plt.colorbar(sc1, ax=axes[1], label="view RMS")

plt.tight_layout(); plt.show()

In [ ]:
WINDOWS = [
    ("GFC",          "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt",      "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue","2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon",  "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID",        "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23",  "2023-03-08", "2023-03-22", "tab:green"),
]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(phi[:, 0], phi[:, 1], c="lightgray", s=6, alpha=0.5, label="all views")
for label, lo, hi, color in WINDOWS:
    mask = (view_dates >= pd.Timestamp(lo)) & (view_dates < pd.Timestamp(hi))
    if mask.sum() == 0:
        continue
    ax.scatter(phi[mask, 0], phi[mask, 1], c=color, s=24, alpha=0.95,
               edgecolors="k", linewidths=0.4, label=f"{label}  (n={mask.sum()})")
ax.set_xlabel("phi_1"); ax.set_ylabel("phi_2")
ax.set_title("Embedding with named crisis windows highlighted")
ax.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()

## 6. Higher dims — does anything beyond `phi_1`, `phi_2` matter?

Pairs plot of the first 4 dims. If `phi_3` / `phi_4` are noise around zero,
useful dimensionality is 2-3. If they show structure (clusters, arcs), keep `d >= 4`.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        if i <= j:
            ax.set_visible(False)
            continue
        ax.scatter(phi[:, j], phi[:, i + 1], c=view_years, cmap="viridis", s=4, alpha=0.6)
        ax.set_xlabel(f"phi_{j + 1}")
        ax.set_ylabel(f"phi_{i + 2}")
plt.suptitle("First 4 embedding dims, pairs colored by year", y=1.02)
plt.tight_layout(); plt.show()

## 7. Nyström out-of-sample extension

The pipeline embeds new views (one per backtest step) via the
`SpectralBasis.embed(v_test)` method. Sanity-check: re-embed the last 200
training views and compare to their actual training embeddings.

(Slight cheat — those points were in the training set; this just confirms
the Nyström wrapper isn't catastrophically broken.)

In [ ]:
n_check = 200
check_idx = np.arange(len(views) - n_check, len(views))
phi_recovered = basis.embed_batch(views[check_idx])
rmse = np.sqrt(((phi_recovered - phi[check_idx]) ** 2).mean(axis=1))
phi_scale = phi.std(axis=0).mean()

print(f"Nystrom-vs-true RMSE on last 200 training views:")
print(f"  median:   {np.median(rmse):.4f}")
print(f"  max:      {rmse.max():.4f}")
print(f"  vs embedding scale (mean std per dim): {phi_scale:.4f}")
print(f"  relative median:  {np.median(rmse) / phi_scale * 100:.1f}% of embedding scale")

## What to look for / what tells you it's working

* **Crisis-window highlights cluster.** Tight blob distinct from quiet-market
  dots → embedding finds regime structure. Random scatter → noise.
* **Year colormap shows arc / drift.** Smooth gradient → slow regime drift
  over years.
* **RMS coloring shows non-trivial layout.** Pure radial gradient would mean
  the embedding is just rotated amplitude; anything more interesting means it
  separates *shapes* of residual trajectories.
* **Nyström RMSE < ~20% of embedding scale.** Confirms the out-of-sample
  wrapper used at test time isn't pathological.